# PaceAlgo ML — Notebook 01: Data Fetch

**Zweck:** OHLCV-Daten für alle Phase-1-Symbole herunterladen, normalisieren und als Parquet cachen.

**Symbole Phase 1 (kostenlos):**
- Crypto: BTCUSDT, ETHUSDT, SOLUSDT — via **KuCoin** (Binance ist US-geo-blocked aus Colab)
- FX: EURUSD, GBPUSD, USDJPY — via Dukascopy
- Metalle: XAUUSD — via Dukascopy
- Macro: VIX, DXY, TNX — via Yahoo Finance

**Hinweis:** Binance Funding Rates / Open Interest aus Phase 1 entfernt (Binance Futures-API ist in Colab US-Region geblockt). Architektur-Entscheidung: Option A — Universal Features tragen Generalisierung, Crypto-Alt-Data wird in Phase 2 ggf. nachgereicht.

**Output:** Parquet-Dateien in `data_cache/raw/<symbol>_<tf>.parquet`

**Laufzeit:** ~30-40 Minuten bei vollem 5-Jahres-Pull.

---

## Bedienung in Colab

1. `File → Open Notebook → GitHub → ecoNC/pace-algo → notebooks/01_fetch_data.ipynb`
2. `Runtime → Run all`
3. Beim ersten Lauf wird das Repo geklont und Dependencies installiert (~2 Min)
4. Daten landen im verbundenen Google Drive unter `MyDrive/pace-algo/data_cache/raw/`


## 1. Environment Setup

In [ ]:
# Detect environment (Colab vs local)
import sys, os
IS_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IS_COLAB}')
print(f'Python: {sys.version.split()[0]}')

In [ ]:
# Mount Google Drive (Colab only) for persistent data cache
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print(f'Project root: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if not PROJECT_ROOT.endswith('pace-algo'):
        PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    print(f'Local project root: {PROJECT_ROOT}')

In [ ]:
# Clone or update repo (always pull latest code from GitHub)
if IS_COLAB:
    if not os.path.exists(os.path.join(PROJECT_ROOT, 'core')):
        !git clone https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
        !cp -r /tmp/pace-algo/. {PROJECT_ROOT}/
        print('Repo cloned to Drive (first time)')
    else:
        # Pull latest code so notebook updates land
        !rm -rf /tmp/pace-algo
        !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
        !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
        print('Core code updated from GitHub')

In [ ]:
# Install dependencies — explicit + verified to catch silent install failures
import subprocess, importlib

REQUIRED = {
    'pandas':            'pandas',
    'pyarrow':           'pyarrow',
    'requests':          'requests',
    'yfinance':          'yfinance',
    'dukascopy_python':  'dukascopy-python',   # import_name : pip_name
    'tqdm':              'tqdm',
}

# Install batch (quiet)
pip_targets = ' '.join(REQUIRED.values())
print(f'Installing: {pip_targets}')
result = subprocess.run(f'pip install -q {pip_targets}', shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print('Pip output:', result.stdout[-500:])
    print('Pip errors:', result.stderr[-500:])

# Verify each can be imported; force-reinstall any that failed
print('\nVerifying imports:')
for import_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(import_name)
        print(f'  OK  {import_name}')
    except ImportError:
        print(f'  FAIL {import_name} — force reinstall')
        ret = subprocess.run(f'pip install --force-reinstall --no-cache-dir {pip_name}',
                              shell=True, capture_output=True, text=True)
        if ret.returncode != 0:
            print(f'    REINSTALL FAILED: {ret.stderr[-300:]}')
        else:
            try:
                importlib.import_module(import_name)
                print(f'  OK  {import_name} (after force reinstall)')
            except ImportError as e:
                print(f'  STILL FAILING: {e}')
print('\nDone.')


In [ ]:
# Import project modules (clearing any stale cached imports)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

from datetime import datetime, timezone
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

from core.config import (
    DATA_RAW, CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES, HTF_CONTEXT_TIMEFRAMES,
    TRAIN_START, TEST_END,
)
from core.data import fetch_klines, save_parquet, fetch_yahoo  # fetch_klines = KuCoin

DATA_RAW.mkdir(parents=True, exist_ok=True)
print(f'Data cache: {DATA_RAW}')
print(f'Date range: {TRAIN_START.date()} → {TEST_END.date()}')
print(f'Crypto symbols: {CRYPTO_SYMBOLS}')
print(f'FX symbols:     {FX_SYMBOLS}')
print(f'Metal symbols:  {METAL_SYMBOLS}')

## 2. Crypto OHLCV (KuCoin) — BTC / ETH / SOL

KuCoin liefert max 1500 Bars/Call. Für 5 Jahre 5M-Daten ≈ 350 Calls pro Symbol/TF.
Bei 0.15s Pause ≈ 1-2 Min pro Symbol/TF. Insgesamt ~15-20 Min für alle 3 Coins × 4 TFs.

In [ ]:
TIMEFRAMES_ALL = PRIMARY_TIMEFRAMES + HTF_CONTEXT_TIMEFRAMES

for symbol in tqdm(CRYPTO_SYMBOLS, desc='Crypto symbols'):
    for tf in TIMEFRAMES_ALL:
        out_path = DATA_RAW / f'{symbol}_{tf}.parquet'
        if out_path.exists():
            print(f'  skip {symbol} {tf} (cached, {out_path.stat().st_size / 1024**2:.1f} MB)')
            continue
        try:
            df = fetch_klines(symbol, tf, TRAIN_START, TEST_END)
            if df.empty:
                print(f'  EMPTY {symbol} {tf}')
                continue
            save_parquet(df, out_path)
            print(f'  OK  {symbol} {tf}: {len(df):,} bars  {df.index[0].date()} → {df.index[-1].date()}')
        except Exception as e:
            print(f'  ERR {symbol} {tf}: {e}')

## 3. FX & Metals (Dukascopy)

In [ ]:
try:
    from core.data.dukascopy_fetcher import fetch_dukascopy_ohlcv
    DUKAS_AVAILABLE = True
    print('Dukascopy fetcher loaded')
except ImportError as e:
    print(f'Dukascopy fetcher unavailable: {e}')
    DUKAS_AVAILABLE = False

In [ ]:
if DUKAS_AVAILABLE:
    for symbol in tqdm(FX_SYMBOLS + METAL_SYMBOLS, desc='FX/Metals'):
        for tf in TIMEFRAMES_ALL:
            out_path = DATA_RAW / f'{symbol}_{tf}.parquet'
            if out_path.exists():
                print(f'  skip {symbol} {tf} (cached, {out_path.stat().st_size / 1024**2:.1f} MB)')
                continue
            try:
                df = fetch_dukascopy_ohlcv(symbol, tf, TRAIN_START, TEST_END)
                if df.empty:
                    print(f'  EMPTY {symbol} {tf}')
                    continue
                save_parquet(df, out_path)
                print(f'  OK  {symbol} {tf}: {len(df):,} bars  {df.index[0].date()} → {df.index[-1].date()}')
            except Exception as e:
                print(f'  ERR {symbol} {tf}: {e}')
else:
    print('Skipping FX/Metals — dukascopy-python not available.')

## 4. Macro Context (Yahoo: VIX, DXY, 10Y Yield)

In [ ]:
out_path = DATA_RAW / 'macro_daily.parquet'
if not out_path.exists():
    try:
        macro = fetch_yahoo(['VIX', 'DXY', 'TNX'], TRAIN_START, TEST_END, interval='1d')
        if not macro.empty:
            save_parquet(macro, out_path)
            print(f'OK macro daily: {len(macro):,} obs  {macro.index[0].date()} → {macro.index[-1].date()}')
            display(macro.tail())
        else:
            print('EMPTY macro')
    except Exception as e:
        print(f'ERR macro: {e}')
else:
    print('Macro daily cached')
    macro = pd.read_parquet(out_path)
    display(macro.tail())

## 5. Sanity Check — BTC 5M Sample

In [ ]:
btc_path = DATA_RAW / 'BTCUSDT_5m.parquet'
if btc_path.exists():
    btc = pd.read_parquet(btc_path)
    print(f'BTC 5M bars: {len(btc):,}')
    print(f'Date range: {btc.index[0]} → {btc.index[-1]}')
    expected = (TEST_END - TRAIN_START).total_seconds() / 300
    coverage = len(btc) / expected * 100
    print(f'Expected (5 years 5M): ~{int(expected):,} bars')
    print(f'Coverage: {coverage:.1f}%')
    print('\nSample tail:')
    display(btc.tail())
else:
    print('BTC 5M not yet fetched')

## 6. Summary

In [ ]:
parquet_files = sorted(DATA_RAW.glob('*.parquet'))
summary = []
for p in parquet_files:
    df = pd.read_parquet(p)
    summary.append({
        'file': p.name,
        'rows': len(df),
        'from': df.index[0] if len(df) else None,
        'to': df.index[-1] if len(df) else None,
        'size_MB': round(p.stat().st_size / 1024**2, 2),
    })
summary_df = pd.DataFrame(summary)
display(summary_df)
print(f'\nTotal cached files: {len(summary)}')
print(f'Total size: {summary_df["size_MB"].sum():.1f} MB')

---

## Nächster Schritt

→ `02_feature_engineering.ipynb`

Berechnet 25-30 ATR-normalisierte Features aus den gecachten OHLCV-Daten.